# Données

## Importation des packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import requests
from bs4 import BeautifulSoup
import os
import s3fs
import ast

## Lecture des fichiers movies_metadata.csv et credits.csv

Les données de movies_metadata.csv et credits.csv sont des données trouvées sur Kaggle qui centralisent des informations diverses sur des films sortis avant juillet 2017. 

Les variables de movies_metadata.csv sont :
adult : signification de la varible non connue
belongs to collection : si le film appartient à une série de film, la variable renseigne les films faisant partie de cette série
budget : budget du film
genres : genres du film
homepage : lien vers le site officiel du film s'il y en a un
id et imdb_id : identifiants du film
original_language : langue originale du film
original_title : titre original du film
overview : résumé du film
popularity : popularité du film sur IMDB
poster_path : lien vers l'affiche du film
production_countries : pays de production du film
production_companies : compagnies de production du film
release_date : date de sortie du film
revenue : recettes du film
runtime : durée du film
spoken_languages : langues parlées dans le film en version originale
status : si le film est sorti, prévu, annulé, en production etc
tagline : catchphrase du film
title : titre anglophone du film
video : False si le film est sorti au cinéma, True s'il est sorti directement sur Internet et qu'il n'a pas été diffusé au cinéma

Les variables de credits.csv sont :
cast : casting du film sous forme de liste de dictionnaires
crew : équipe du film sous forme de liste de dictionnaires

Nous souhaitons à partir de ces données prédire la note de nouveaux films, voir quelles sont les variables les plus décisives pour prédire si un film sera ien reçu par le public et ainsi remarquer (ou non) la prévisibilité du succès d'un film.

Nous pourrons pondérer l'erreur de prévision avec la variable vote_count et faire de la classification non supervisée dans les stats descriptives

In [2]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_KEY_S3 = '/movies_metadata.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_movies = pd.read_csv(file_in,sep=',', header=0)

/tmp/ipykernel_13749/2309787388.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_movies = pd.read_csv(file_in,sep=',', header=0)


In [3]:
FILE_KEY_S3 = '/credits.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_credits = pd.read_csv(file_in,sep=',', header=0)

Après première exploration des données, nous avons décidé d'enlever les variables suivantes : adult, homepage, overview, popularity, poster_path, spoken_languages et tagline. En effet, la variable adult présente presque toujours la modalité False et semble avoir peu d'intérêt. La variable homepage renseigne le lien vers le site officiel du film s'il y en a un. La variable overview contient les résumés des films du dataframe, ce qui peut être intéressant à exploiter mais nous avons décidé de ne pas le faire. La variable popularity est la popularité du film sur IMDB au moment où les données ont été extraites, c'est donc une variable qui n'est pas statique et qui est calculée directement par IMDB d'une façon que nous ignorons donc nous ne souhaitons pas la prendre en compte. La variable poster_path indique le lien vers l'affiche du film, nous n'en avons pas besoin. La variable spoken_languages indique les langues parlées durant le film en version originale, nous considérons que cette variable est redondante par rapport à la variable original_language. Enfin la variable tagline indique la catchphrase du film, ce qui est à nos yeux peu utile également.

Nous allons retraiter certaines variables. Par exemple, la variable belongs_to_collection sera transformée en booléen (1 si le film correspond à une série de films, 0 sinon) à laquelle nous ajouterons une variable avec le nombre de films précédents de la série ainsi que la note du film précédent. La variable genres sera décomposée en plusieurs variables genre_1, genre_2 etc. Ce genre de décomposition sera également nécessaire pour les variables production_countries et production_companies

Les variables budget et runtime présentent des valeurs manquantes, que nous allons essayer de compléter avec du web scraping.

Nous allons nous concentrer sur les films qui sont déjà sortis en salle (status = Released et video=False)

Les données du fichier credits.csv vont nous permettre d'ajouter les acteurs principaux et le réalisateur du film à notre jeu de données. Nous souhaitons ajouter des variables relatives à la popularité des acteurs et du réalisateur via du web scraping.


In [4]:
data_movies.original_language.value_counts()

original_language
en      32269
fr       2438
it       1529
ja       1350
de       1080
        ...  
sm          1
82.0        1
hy          1
lb          1
si          1
Name: count, Length: 92, dtype: int64

In [26]:
data_movies_df = data_movies[data_movies['video'] == False]
data_movies_df = data_movies_df[data_movies_df['status'] == 'Released']
data_movies_df = data_movies_df.drop(columns=['adult', 'homepage', 'overview', 'popularity', 'poster_path', 'tagline', 'status', 'video'])
data_movies_df = data_movies_df.dropna(subset= ['release_date'])
data_movies_df = data_movies_df.dropna(subset= ['imdb_id'])
data_movies_df = data_movies_df.dropna(subset= ['original_language'])

In [6]:
data_movies_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44825 entries, 0 to 45465
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   belongs_to_collection  4460 non-null   object 
 1   budget                 44825 non-null  object 
 2   genres                 44825 non-null  object 
 3   id                     44825 non-null  object 
 4   imdb_id                44825 non-null  object 
 5   original_language      44825 non-null  object 
 6   original_title         44825 non-null  object 
 7   production_companies   44825 non-null  object 
 8   production_countries   44825 non-null  object 
 9   release_date           44825 non-null  object 
 10  revenue                44825 non-null  float64
 11  runtime                44588 non-null  float64
 12  title                  44825 non-null  object 
 13  vote_average           44825 non-null  float64
 14  vote_count             44825 non-null  float64
dtypes: floa

In [7]:
missing_percentage = data_movies_df.isna().sum()

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

MISSING VALUES :
belongs_to_collection    40365
runtime                    237
dtype: int64


On ajoute au dataframe les différents url wikipédia possibles pour un film (selon le nom du film, il faut parfois ajouter film ou film + année de sortie à l'url wikipédia pour tomber sur la bonne page wiki)

In [27]:
url_wikipedia_fr = "https://fr.wikipedia.org/wiki/"
url_wikipedia_en = "https://en.wikipedia.org/wiki/"
data_movies_df['url'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_")
data_movies_df['url_film'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_") + "_(film)"
data_movies_df['release_year'] = data_movies_df.release_date.str[:4]
data_movies_df['url_film_date'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_") + "_(film,_" + data_movies_df.release_year + ")"
data_movies_df['id'] = pd.to_numeric(data_movies_df['id'])


On joint les dataframes movies et credits pour ajouter le casting et l'équipe du film.

In [28]:
data_movies_credits = data_movies_df.merge(data_credits, left_on='id', right_on='id')
data_movies_credits


,belongs_to_collection,budget,genres,id,imdb_id,original_language,original_title,production_companies,production_countries,release_date,...,spoken_languages,title,vote_average,vote_count,url,url_film,release_year,url_film_date,cast,crew
0,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,tt0114709,en,Toy Story,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Toy Story,7.7,5415.0,https://en.wikipedia.org/wiki/Toy_Story,https://en.wikipedia.org/wiki/Toy_Story_(film),1995,"https://en.wikipedia.org/wiki/Toy_Story_(film,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,tt0113497,en,Jumanji,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,...,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Jumanji,6.9,2413.0,https://en.wikipedia.org/wiki/Jumanji,https://en.wikipedia.org/wiki/Jumanji_(film),1995,"https://en.wikipedia.org/wiki/Jumanji_(film,_1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,tt0113228,en,Grumpier Old Men,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Grumpier Old Men,6.5,92.0,https://en.wikipedia.org/wiki/Grumpier_Old_Men,https://en.wikipedia.org/wiki/Grumpier_Old_Men...,1995,https://en.wikipedia.org/wiki/Grumpier_Old_Men...,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,tt0114885,en,Waiting to Exhale,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Waiting to Exhale,6.1,34.0,https://en.wikipedia.org/wiki/Waiting_to_Exhale,https://en.wikipedia.org/wiki/Waiting_to_Exhal...,1995,https://en.wikipedia.org/wiki/Waiting_to_Exhal...,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",11862,tt0113041,en,Father of the Bride Part II,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Father of the Bride Part II,5.7,173.0,https://en.wikipedia.org/wiki/Father_of_the_Br...,https://en.wikipedia.org/wiki/Father_of_the_Br...,1995,https://en.wikipedia.org/wiki/Father_of_the_Br...,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44893,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 28, 'name...",30840,tt0102797,en,Robin Hood,"[{'name': 'Westdeutscher Rundfunk (WDR)', 'id'...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",1991-05-13,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Robin Hood,5.7,26.0,https://en.wikipedia.org/wiki/Robin_Hood,https://en.wikipedia.org/wiki/Robin_Hood_(film),1991,https://en.wikipedia.org/wiki/Robin_Hood_(film...,"[{'cast_id': 1, 'character': 'Sir Robert Hode'...","[{'credit_id': '52fe44439251416c9100a899', 'de..."
44894,NaN,0,"[{'id': 18, 'name': 'Drama'}]",111109,tt2028550,tl,Siglo ng Pagluluwal,"[{'name': 'Sine Olivia', 'id': 19653}]","[{'iso_3166_1': 'PH', 'name': 'Philippines'}]",2011-11-17,...,"[{'iso_639_1': 'tl', 'name': ''}]",Century of Birthing,9.0,3.0,https://en.wikipedia.org/wiki/Century_of_Birthing,https://en.wikipedia

On retraite les colonnes cast et crew pour que Python les reconnaissent en tant que liste de dictionnaires.

In [29]:
data_movies_credits['cast'] = data_movies_credits['cast'].apply(ast.literal_eval)
data_movies_credits['crew'] = data_movies_credits['crew'].apply(ast.literal_eval)

On ajoute les colonnes correspondant aux 4 acteurs principaux du film et une colonne pour le réalisateur du film.

In [30]:
data_movies_credits['acteur_1'] = data_movies_credits['cast'].apply(
    lambda lst: lst[0]['name'] if isinstance(lst, list) and len(lst) > 0 else None
)
data_movies_credits['acteur_2'] = data_movies_credits['cast'].apply(
    lambda lst: lst[1]['name'] if isinstance(lst, list) and len(lst) > 1 else None
)
data_movies_credits['acteur_3'] = data_movies_credits['cast'].apply(
    lambda lst: lst[2]['name'] if isinstance(lst, list) and len(lst) > 2 else None
)
data_movies_credits['acteur_4'] = data_movies_credits['cast'].apply(
    lambda lst: lst[3]['name'] if isinstance(lst, list) and len(lst) > 3 else None
)

data_movies_credits['realisateur'] = data_movies_credits['crew'].apply(
    lambda lst: lst['job' == 'Director']['name'] if isinstance(lst, list) and len(lst) > 0 else None
)


On enlève les lignes où il n'y a pas d'acteurs.

In [31]:
data_movies_credits = data_movies_credits[data_movies_credits['cast'].apply(lambda x: len(x) != 0)]
data_movies_credits

,belongs_to_collection,budget,genres,id,imdb_id,original_language,original_title,production_companies,production_countries,release_date,...,url_film,release_year,url_film_date,cast,crew,acteur_1,acteur_2,acteur_3,acteur_4,realisateur
0,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,tt0114709,en,Toy Story,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,...,https://en.wikipedia.org/wiki/Toy_Story_(film),1995,"https://en.wikipedia.org/wiki/Toy_Story_(film,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",Tom Hanks,Tim Allen,Don Rickles,Jim Varney,John Lasseter
1,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,tt0113497,en,Jumanji,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,...,https://en.wikipedia.org/wiki/Jumanji_(film),1995,"https://en.wikipedia.org/wiki/Jumanji_(film,_1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",Robin Williams,Jonathan Hyde,Kirsten Dunst,Bradley Pierce,Larry J. Franco
2,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,tt0113228,en,Grumpier Old Men,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,...,https://en.wikipedia.org/wiki/Grumpier_Old_Men...,1995,https://en.wikipedia.org/wiki/Grumpier_Old_Men...,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",Walter Matthau,Jack Lemmon,Ann-Margret,Sophia Loren,Howard Deutch
3,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,tt0114885,en,Waiting to Exhale,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,...,https://en.wikipedia.org/wiki/Waiting_to_Exhal...,1995,https://en.wikipedia.org/wiki/Waiting_to_Exhal...,"[{'cast_id': 1, 'character': 'Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",Whitney Houston,Angela Bassett,Loretta Devine,Lela Rochon,Forest Whitaker
4,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",11862,tt0113041,en,Father of the Bride Part II,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,...,https://en.wikipedia.org/wiki/Father_of_the_Br...,1995,https://en.wikipedia.org/wiki/Father_of_the_Br...,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",Steve Martin,Diane Keaton,Martin Short,Kimberly Williams-Paisley,Alan Silvestri
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44892,NaN,0,"[{'id': 878, 'name': 'Science Fiction'}]",222848,tt0112613,en,Caged Heat 3000,"[{'name': 'Concorde-New Horizons', 'id': 4688}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-01-01,...,https://en.wikipedia.org/wiki/Caged_Heat_3000_...,1995,https://en.wikipedia.org/wiki/Caged_Heat_3000_...,"[{'cast_id': 1, 'character': 'Kira (as Cassand...","[{'credit_id': '5757f36ac3a3687d6f000e8a', 'de...",Lisa Boyle,Kena Land,Zaneta Polard,Don Yanan,Roger Corman
44893,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 28, 'name...",30840,tt0102797,en,Robin Hood,"[{'name': 'Westdeutscher Rundfunk (WDR)', 'id'...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",1991-05-13,...,https://en.wikipedia.org/wiki/Robin_Hood_(film),1991,https://en.wikipedia.org/wiki/Robin_Hood_(film...,"[{'cast_id': 1, 'character': 'Sir Robert Hode'...","[{'credit_id': '52fe44439251416c9100a899', 'de...",Patrick Bergin,Uma Thurman,David Morrissey,Jürgen Prochnow,John Irvin
44894,NaN,0,"[{'id': 18, 'name': 'Drama'}]",111109,tt2028550,tl,Siglo ng Pagluluwal,"[{'name': 'Sine Olivia', 'i

In [49]:
data=data_movies_credits

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
}

Fonction pour trouver le budget d'un film avec l'url wikipédia

In [50]:
def extraire_budget_depuis_wikipedia(url):
    try:
        # Charger la page
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parser le HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # Trouver l'infobox (il peut y avoir plusieurs classes, mais 'infobox' est souvent commun)
        infobox = soup.find('table', class_='infobox')

        if infobox is None:
            return None  # Pas d'infobox trouvée

        # Chercher les lignes de l'infobox
        rows = infobox.find_all('tr')

        for row in rows:
            header = row.find('th')
            if header and 'budget' in header.get_text(strip=True).lower():
                # Trouver la cellule contenant la valeur
                value_cell = row.find('td')
                if value_cell:
                    return value_cell.get_text(separator=" ", strip=True)

        return None  # Pas de ligne contenant "budget"

    except Exception as e:
        print(f"Erreur lors du traitement de {url}: {e}")
        return None


In [52]:
data['budget_2'] = data['url'].apply(extraire_budget_depuis_wikipedia)


Erreur lors du traitement de https://en.wikipedia.org/wiki/How_To_Make_An_American_Quilt: 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/How_To_Make_An_American_Quilt
Erreur lors du traitement de https://en.wikipedia.org/wiki/Nico_Icon: 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/Nico_Icon
Erreur lors du traitement de https://en.wikipedia.org/wiki/The_Neverending_Story_III:_Escape_from_Fantasia: 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/The_Neverending_Story_III:_Escape_from_Fantasia
Erreur lors du traitement de https://en.wikipedia.org/wiki/Jupiter's_Wife: 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/Jupiter's_Wife
Erreur lors du traitement de https://en.wikipedia.org/wiki/Sonic_Outlaws: 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/Sonic_Outlaws
Erreur lors du traitement de https://en.wikipedia.org/wiki/Free_Willy_2_-_The_Adventure_Home: 404 Client Error: Not Found for url: 

/tmp/ipykernel_13749/3722422632.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['budget_2'] = data['url'].apply(extraire_budget_depuis_wikipedia)


In [61]:
data[['title','budget', 'budget_2']].head(500)

,title,budget,budget_2
0,Toy Story,30000000,$30 million [ 2 ]
1,Jumanji,65000000,$65 million [ 1 ]
2,Grumpier Old Men,0,$25 million
3,Waiting to Exhale,16000000,$16 million
4,Father of the Bride Part II,0,<$40 million [ 1 ]
5,Heat,60000000,None
6,Sabrina,58000000,None
7,Tom and Huck,0,None
8,Sudden Death,35000000,None
9,GoldenEye,58000000,$60 million [ 3 ]


In [60]:
pd.set_option('display.max_rows', 500)

In [56]:
# Ton DataFrame à sauvegarder
# Exemple : df = pd.DataFrame({'col1': [1, 2], 'col2': ['a', 'b']})
BUCKET = 'mlepennec-ensae'

FILE_OUT_S3 = '/movies_export.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    data.to_csv(f_out, index=False)


In [148]:

if r_url_film.status_code == 404:
    print("url simple ne fonctionne pas")
    liste_acteurs.append("impossible de récupérer la liste des acteurs")
else:
    print("pas erreur 404")
    soup_film = BeautifulSoup(r_url_film.text, "html.parser")

    selector_homonymie = "div:nth-child(2) > p > a"
    homonymie = soup_film.select(selector_homonymie)

    if homonymie[0].text=='page d’homonymie':
        print("homonymie")
        url_film = data_movies_df.url_film_date[i]
        
        r_url_film = requests.get(url_film)

        if r_url_film.status_code == 404:
            print("url avec film et annee ne fonctionne pas")
            url_film = data_movies_df.url_film[i]
            r_url_film = requests.get(url_film)

            if r_url_film.status_code == 404:
                print("url avec annee ne fonctionne pas")
                liste_acteurs.append("impossible de récupérer la liste des acteurs")
            else:
                soup_film = BeautifulSoup(r_url_film.text, "html.parser")

                selector_film = "div.infobox_v3 table tr td div p"
                acteurs = soup_film.select(selector_film)
        
                liste_acteurs.append(acteurs[0].text)
    else:
        print("pas d'homonymie")
        soup_film = BeautifulSoup(r_url_film.text, "html.parser")

        selector_film = "div.infobox_v3 table tr td div p"
        acteurs = soup_film.select(selector_film)
        if len(acteurs)==0:
            print("essai d'autres url")
            url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film,_" + data.release_date[i][0:4] + ")"
        
            r_url_film = requests.get(url_film)

            if r_url_film.status_code == 404:

                url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film)"
                r_url_film = requests.get(url_film)

                if r_url_film.status_code == 404:

                    liste_acteurs.append("impossible de récupérer la liste des acteurs")
                else:
                    soup_film = BeautifulSoup(r_url_film.text, "html.parser")

                    selector_film = "div.infobox_v3 table tr td div p"
                    acteurs = soup_film.select(selector_film)
        
                    liste_acteurs.append(acteurs[0].text)
        else:
            liste_acteurs.append(acteurs[0].text)

pas erreur 404
homonymie


In [183]:
url_film

'https://fr.wikipedia.org/wiki/Balto_(film)'

In [135]:
url_wikipedia = "https://fr.wikipedia.org/wiki/"
liste_acteurs=[]
for i in range(100):
    print(i)
    url_film = url_wikipedia + data.title[i].replace(" ", "_")

    r_url_film = requests.get(url_film, headers = headers)
    if r_url_film.status_code == 404:
        liste_acteurs.append("impossible de récupérer la liste des acteurs")
    else:
        soup_film = BeautifulSoup(r_url_film.text, "html.parser")

        selector_film = "div div p a"
        acteurs = soup_film.select(selector_film)

        if acteurs[0].text=='page d’homonymie':
            url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film,_" + data.release_date[i][0:4] + ")"
        
            r_url_film = requests.get(url_film, headers = headers)

            if r_url_film.status_code == 404:

                url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film)"
                r_url_film = requests.get(url_film, headers = headers)

                if r_url_film.status_code == 404:

                    liste_acteurs.append("impossible de récupérer la liste des acteurs")
                else:
                    soup_film = BeautifulSoup(r_url_film.text, "html.parser")

                    selector_film = "div.infobox_v3 table tr td div p"
                    acteurs = soup_film.select(selector_film)
        
                    liste_acteurs.append(acteurs[0].text)
        else:
            soup_film = BeautifulSoup(r_url_film.text, "html.parser")

            selector_film = "div.infobox_v3 table tr td div p"
            acteurs = soup_film.select(selector_film)
            if len(acteurs)==0:
                print("essai d'autres url")
                url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film,_" + data.release_date[i][0:4] + ")"
        
                r_url_film = requests.get(url_film, headers = headers)

                if r_url_film.status_code == 404:

                    url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film)"
                    r_url_film = requests.get(url_film, headers = headers)

                    if r_url_film.status_code == 404:

                        liste_acteurs.append("impossible de récupérer la liste des acteurs")
                    else:
                        soup_film = BeautifulSoup(r_url_film.text, "html.parser")

                        selector_film = "div.infobox_v3 table tr td div p"
                        acteurs = soup_film.select(selector_film)
        
                        liste_acteurs.append(acteurs[0].text)
            else:
                liste_acteurs.append(acteurs[0].text)

0


1
2
3
4
5
6
7
8
9
10
11
12
essai d'autres url
13
14
essai d'autres url
15
16
essai d'autres url
17
18
19
20
21
22
essai d'autres url
23
24
25
26
27
essai d'autres url
28
29
30
31
32
33
essai d'autres url
34
35
36
37
38
39
essai d'autres url
40
essai d'autres url
41
essai d'autres url
42


IndexError: list index out of range

In [136]:
liste_acteurs

['Tom HanksTim Allen\n',
 'impossible de récupérer la liste des acteurs',
 'Whitney Houston Angela BassettLoretta Devine  Lela Rochon\n',
 'impossible de récupérer la liste des acteurs',
 'impossible de récupérer la liste des acteurs',
 'impossible de récupérer la liste des acteurs',
 'Pierce BrosnanSean BeanIzabella ScorupcoFamke JanssenAlan Cumming\n',
 'Michael DouglasAnnette BeningMartin SheenMichael J. Fox\n',
 'impossible de récupérer la liste des acteurs',
 'Kevin BaconBob HoskinsBridget FondaJim CummingsPhil Collins\n',
 'Anthony HopkinsJoan AllenPowers BootheEd Harris\n',
 'Geena DavisMatthew Modine Frank LangellaMaury ChaykinPatrick Malahide\n',
 'Robert De NiroJoe PesciSharon Stone\n',
 'impossible de récupérer la liste des acteurs',
 'Tim Roth  Madonna  Valeria Golino  Jennifer Beals  Antonio Banderas\n',
 'Jim CarreyIan McNeiceSimon CallowBob GuntonMaynard Eziashi\n',
 'Wesley SnipesWoody HarrelsonJennifer LopezRobert Blake\n',
 'John TravoltaGene HackmanRene RussoDanny De